## Embeddings and Embedding Models

## Prerequisite Understanding:
We are now going towards learning how RAG works.<br>
Basically, when we as users provide documents to an AI, how does an AI understand from which part of this big document should it provide relevent answers to us users.<br>
First let us understand the meaning of 'vector embeddings':<br>
vector -> a list of numbers <br>
embeddings -> the process of converting text into that list of numbers<br>
So, <br>
vector embeddings -> a list of numbers that represents the meaning of a piece of text<br>
.........................................................<br>
The problem we're solving:<br>
Suppose we have 148 chunks of text. When a user asks a question, how do we know which chunks are relevant to answer it?<br>
We can't send all 148 chunks to the AI, cause its too expensive and slow. We need to find the 2-3 most relevant ones.<br>
That's what embeddings and vector stores solve.<br>
.........................................................<br>
Here is how it works:
1. We load our DOC into the Document Object.
2. We Split it into chunks.
3. Each chunk(set of texts) is converted into numbers (embeddings) and stored in a database 'Chromadb'.
4. When a user asks a question such as "What is a chain in LangChain?"
5. That question gets converted to numbers too..
6. Chroma finds which chunks in it have the most similar numbers (similar vectors in it, is comparison to vectors of the given question)
7. Those chunks get sent to the AI
8. AI reads those specific chunks and answers the question

.........................................................<br>
In Easy words:
So embeddings is a list of numbers..
And each chunk(a particular set of texts) has an 'embedding' stored in Chromadb (where embedding is like a coordinate in the chroma database map). And when a new question is asked by the user, that's also turned into a list of numbers, which is taken in like an embedding to chromadb, and gives results related to other chunks lying around this new embedding.<br>
.........................................................<br>
Example of how an embedding looks like:<br>
(suppose we have 3 chunks:)<br>
"LangChain is a framework" → [0.23, -0.51, 0.87, 0.12, ...]<br>
"Python is a language"     → [0.19, -0.48, 0.91, 0.09, ...]<br>
"I love pizza"             → [-0.72, 0.33, -0.15, 0.61, ...]<br>
The key insight is that "similar meaning = similar numbers".<br>
The sentences "LangChain is a framework" and "Python is a language" are both about programming, so their vectors are close to each other.<br> "I love pizza" is completely different, so its vector is far away. <br>
[Don't see the numbers, we won't understand like that. Just imagine Chromadb like a x-y graph, where these embeddings are like 'coordinates'. Chunks/Texts having similar meaning lie near each other in the graph, or in ChromaDB.]<br>
ChromaDB -> It's a vector database designed to store and search vectors efficiently. Regular databases search by exact match. Chroma searches by similarity. Its like: "find me the closest vectors to this query vector."<br>
.........................................................<br>
PDF → chunks → embeddings → stored in Chroma<br>
User question → embedding → similarity search in Chroma → relevant chunks → AI → answer<br>
.........................................................<br>
Now, before moving ahead, let's understand how chunks are stored in ChromaDB.<br>
These chunks can be anything from the given doc: simple phrases, sentences, or even big paragraphs.<br>
Anyways, the whole chunk is converted into a single vector, not individual parts from the chunk.<br>
.........................................................<br>
What about Ambiguous words? <br>
Because if that's the case, a chunk can also be a big paragraph..sometimes big enough that a single 'word' can even have 2 meanings in itself.Something like: "I went to the bank to deposit money, then walked along the river bank to relax."<br>
Both meanings of "bank" are in the same chunk, so what does the vector look like?<br>
In such a situation, The vector becomes a blend of both meanings. It gets pulled in both directions. <br>
Yes, this can cause problems sometimes. This is a real limitation of embeddings called the "semantic blur" problem.<br>
If someone searches "river bank" this chunk might show up. If someone searches "financial bank" this chunk might also show up. It's not perfectly precise.<br>
How real RAG systems handle this:<br>
1. Smaller chunks — less chance of two conflicting meanings in one chunk<br>
2. Better embedding models — newer models handle context and ambiguity much better<br>
3. Reranking — after similarity search, a second model re-reads the actual text and reranks results by true relevance<br>

.........................................................<br>
Also, problems with the size of chunks can arrive:<br>
Small chunk: "LangChain builds AI apps"  → very specific vector, easy to match<br>
Big chunk: "LangChain builds AI apps. I love pizza. The weather is nice." → confused vector, harder to match accurately<br>
This is exactly why chunk size matters so much in RAG. If its too big, the embedding loses focus. And if its too small, it loses context.<br>
The sweet spot is usually chunks that cover one idea or topic, so not too broad, nor too narrow.<br>
.........................................................<br>

In [1]:
%%capture
!pip install --force-reinstall --no-cache-dir tenacity==8.2.3 --user
!pip install "ibm-watsonx-ai==1.0.8" --user
!pip install "ibm-watson-machine-learning==1.0.367" --user
!pip install "langchain-ibm==0.1.7" --user
!pip install "langchain-community==0.2.10" --user
!pip install "langchain-experimental==0.0.62" --user
!pip install "langchainhub==0.1.18" --user
!pip install "langchain==0.2.11" --user
!pip install "pypdf==4.2.0" --user
!pip install "chromadb==0.4.24" --user

In [ ]:
import os
os._exit(00)

In [1]:
# We can also use this section to suppress warnings generated by your code:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')
import os
os.environ['ANONYMIZED_TELEMETRY'] = 'False'

from ibm_watsonx_ai.foundation_models import ModelInference
from ibm_watsonx_ai.metanames import GenTextParamsMetaNames as GenParams
from ibm_watsonx_ai.foundation_models.utils.enums import ModelTypes
from ibm_watson_machine_learning.foundation_models.extensions.langchain import WatsonxLLM

Loading the Doc into a Document Object:

In [2]:
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/96-FDF8f7coh0ooim7NyEQ/langchain-paper.pdf")
document = loader.load()

In [3]:
from langchain.text_splitter import CharacterTextSplitter
text_splitter = CharacterTextSplitter(chunk_size=200, chunk_overlap=20, separator="\n")
chunks_container = text_splitter.split_documents(document)
print(len(chunks_container)) #To find how many chunks were made
#Here 'chunks_container' variable contains all chunks created

148


## Now, working upon Embeddings from here:

In [4]:
# Import the EmbedTextParamsMetaNames class from ibm_watsonx_ai.metanames module
# This class provides constants for configuring Watson embedding parameters
from ibm_watsonx_ai.metanames import EmbedTextParamsMetaNames

# Configure embedding parameters using a dictionary:
# - TRUNCATE_INPUT_TOKENS: Limits the input to 3 tokens (very short for testing). 
# - RETURN_OPTIONS: Request IBM that the original input text be returned along with embeddings. Useful for Debugging, as we can verify which text produced which vector.
embed_params = {
 EmbedTextParamsMetaNames.TRUNCATE_INPUT_TOKENS: 3,
 EmbedTextParamsMetaNames.RETURN_OPTIONS: {"input_text": True},
}
#'TRUNCATE_INPUT_TOKENS' refers to how much tokens of the given chunk must the AI consider to translate. Here, 3 tokens means roughly the first 3 words of the given chunk texts.
#'RETURN_OPTIONS' asks the AI to return even the original words with its embeddings (to know which token was converted into what embedding).

In [5]:
# Import the WatsonxEmbeddings class from langchain_ibm module
# This provides an integration between LangChain and IBM's Watson AI services
from langchain_ibm import WatsonxEmbeddings

# Create a WatsonxEmbeddings instance with the following configuration:
# - model_id: Specifies the "slate-125m-english-rtrvr-v2" embedding model from IBM
# - url: The endpoint URL for the Watson service in the US South region
# - project_id: The Watson project ID to use ("skills-network")
# - params: The embedding parameters configured earlier
watsonx_embedding = WatsonxEmbeddings(
    model_id="ibm/granite-embedding-278m-multilingual", #This is IBM's Embedding Model. OpenAI, Hugging Face, and others offer embedding models too. Here, we will use the embedding model from IBM's watsonx.ai to work with the text.
    url="https://us-south.ml.cloud.ibm.com",
    project_id="skills-network",
    params=embed_params,
)
#Just the connection to IBM's embedding model (same concept as connecting to the LLM before, but this model's job is specifically to convert text to vectors, not to generate text).

In [6]:
texts = [text.page_content for text in chunks_container] #For every chunk in chunk_container, put all text content from 148 chunks in 'texts' (leave the metadata).
# This is a list comprehension. It loops through all 148 chunks and pulls out just the text content from each one
embedding_result = watsonx_embedding.embed_documents(texts) #This sends all 148 texts to IBM's embedding model and gets back 148 vectors
embedding_result[0][:5] # '0' is first chunk's vector | ':5' is just the first 5 numbers out of hundreds

[-0.04104941338300705,
 0.01380295492708683,
 -0.05267169326543808,
 0.011825969442725182,
 0.03604104742407799]

In [7]:
# Clean up the first text chunk by removing messy line breaks
clean_first_text = texts[0].replace('\n', ' ').strip()

# Grab just the first 3 numbers of its embedding
first_embedding_values = embedding_result[0][:3]

print(f"--- FIRST CHUNK ONLY ---")
print(f"Text Snippet:  {clean_first_text[:60]}...") #slices the text to keep only the first 60 characters (letters, spaces, and punctuation) and discards the rest.
print(f"Embeddings:    {first_embedding_values}")

--- FIRST CHUNK ONLY ---
Text Snippet:  * corresponding author - jkim72@kent.edu  Revolutionizing Me...
Embeddings:    [-0.04104941338300705, 0.01380295492708683, -0.05267169326543808]


## Vector stores:
A vector store (or vector database) is a specialized database designed to store, organize, and search through AI numbers (embeddings). In traditional databases (like Excel or SQL), we search for exact words. In a vector store, we search by meaning and concepts.
When we build an AI application (like a chatbot), we cannot feed a massive 500-page medical textbook directly into the AI every time a user asks a question. 
It is too slow and too expensive.Instead, we use a vector store to do this:You ask a question: "How do I deal with panic attacks?"The chatbot converts your question into numbers (an embedding).The Vector Store searches its map: It instantly finds the 2 or 3 text chunks that are physically closest to your question's numbers.The chatbot reads only those 3 chunks to give you an accurate, instant answer.This entire architecture is called RAG (Retrieval-Augmented Generation).

In [8]:
from langchain.vectorstores import Chroma

In [9]:
docsearch = Chroma.from_documents(chunks_container, watsonx_embedding)
#Takes all 148 chunks -> Converts each one to a vector using watsonx_embedding -> Stores both the vector AND the original chunk in Chroma.

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [12]:
query = "Langchain"
docs = docsearch.similarity_search(query)
# This Takes our query "LangChain" -> Converts it to a vector -> Compares that vector to all 148 stored vectors -> Returns the chunks whose vectors are closest.
# similarity_search() returns a list of the most relevant chunks it found. By default it returns the top 4.
print(docs[0].page_content) #Just prints the text of the most similar chunk found.

Lang Chain's ChatPrompt Template, HumanMessage  Prompt 
Template, ConversationBufferMemory, and LLMChain, 
creating an advanced solution for early detection and


So here, we're not searching for the word "LangChain" literally. We're searching by meaning. So even chunks that don't contain the exact word "LangChain" but discuss similar concepts could still show up.<br>
NOTE: We can safely ignore the warnings related to telemetry events. They are related to ChromaDB's telemetry collection system and do not affect the functionality of our code. Our vector search and similarity operations will work correctly despite these messages.

In [14]:
#To see all 4 results:
for i, doc in enumerate(docs):
    print(f"=== Result {i+1} ===")
    print(doc.page_content)
    print()

=== Result 1 ===
Lang Chain's ChatPrompt Template, HumanMessage  Prompt 
Template, ConversationBufferMemory, and LLMChain, 
creating an advanced solution for early detection and

=== Result 2 ===
LangChain helps us to unlock the ability to harness the 
LLM’s immense potential in tasks such as document analysis, 
chatbot development, code analysis, and countless other

=== Result 3 ===
LangChain provides a lot of utilities for adding memory to a system. These utilities can be used by themselves or 
incorporated seamlessly into a chain.  
A memory system must support two fundamental

=== Result 4 ===
keeping a sliding window of the most recent 
interactions, so the buffer does not get too large . 
The MindGuide chatbot  uses conversation buffer memory.



## Retrievers